In [ ]:
!pip uninstall -y torch torchvision torchaudio dgl torchdata

!pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121
!pip install dgl -f https://data.dgl.ai/wheels/torch-2.3/cu121/repo.html
!pip install torchdata==0.7.1 tensorboardX tqdm
!pip install ogb

import os
import time
import random
import numpy as np
import json
import argparse
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import dgl
from dgl.dataloading import GraphDataLoader
from tensorboardX import SummaryWriter
from tqdm import tqdm
from ogb.graphproppred import DglGraphPropPredDataset, Evaluator
from ogb.graphproppred.mol_encoder import AtomEncoder, BondEncoder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: torchdata 0.11.0
Uninstalling torchdata-0.11.0:
  Successfully uninstalled torchdata-0.11.0
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.9/780.9 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 25.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 115.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 81.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 55.9 MB/s eta 0:00:00
    

DGL backend not selected or invalid.  Assuming PyTorch for now.


Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)


In [ ]:
def compute_avg_degree(dataset):
    log_degrees = []

    for g, _ in dataset:
        d = g.in_degrees().float()
        log_degrees.append(torch.log(d + 1.0))

    all_log_degrees = torch.cat(log_degrees)
    avg_log_d = all_log_degrees.mean().item()

    return avg_log_d

In [ ]:
class FCLayer(nn.Module):
    def __init__(self, in_size, out_size, activation='relu'):
        super().__init__()
        self.linear = nn.Linear(in_size, out_size)

        if activation.lower() == 'relu':
            self.act = nn.ReLU()
        elif activation.lower() == 'leakyrelu':
            self.act = nn.LeakyReLU()
        else:
            self.act = nn.Identity()

    def forward(self, x):
        return self.act(self.linear(x))

class MLP(nn.Module):
    def __init__(self, in_size, hidden_size, out_size, layers, mid_activation='relu'):
        super().__init__()
        self.net = nn.ModuleList()
        if layers == 1:
            self.net.append(nn.Linear(in_size, out_size))
        else:
            self.net.append(FCLayer(in_size, hidden_size, mid_activation))
            for _ in range(layers - 2):
                self.net.append(FCLayer(hidden_size, hidden_size, mid_activation))
            self.net.append(nn.Linear(hidden_size, out_size))

    def forward(self, x):
        for layer in self.net:
            x = layer(x)
        return x

In [ ]:
class PNATower(nn.Module):
    def __init__(self, in_features, out_features, aggregators, scalers, avg_log_d,
                 pretrans_layers=1, posttrans_layers=1, edge_dim=0,
                 use_batch_norm=True, use_graph_norm=True, dropout=0.0):
        super().__init__()
        self.aggregators = aggregators
        self.scalers = scalers
        self.avg_log_d = avg_log_d
        self.use_edge_feat = edge_dim > 0

        pretrans_in_dim = 2 * in_features + edge_dim
        self.pretrans = MLP(in_size=pretrans_in_dim, hidden_size=in_features,
                            out_size=in_features, layers=pretrans_layers)

        aggr_dim = len(aggregators) * len(scalers) * in_features
        self.posttrans = MLP(in_size=in_features + aggr_dim, hidden_size=out_features,
                             out_size=out_features, layers=posttrans_layers)

        self.use_graph_norm = use_graph_norm
        self.use_batch_norm = use_batch_norm
        if use_batch_norm:
            self.batch_norm = nn.BatchNorm1d(out_features)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, g, h, snorm_n, e=None):
        with g.local_scope():
            g.ndata['h'] = h
            if self.use_edge_feat and e is not None:
                g.edata['e'] = e

            def message_func(edges):
                if self.use_edge_feat and e is not None:
                    h_cat = torch.cat([edges.src['h'], edges.dst['h'], edges.data['e']], dim=-1)
                else:
                    h_cat = torch.cat([edges.src['h'], edges.dst['h']], dim=-1)
                return {'msg': self.pretrans(h_cat)}

            def reduce_func(nodes):
                msg = nodes.mailbox['msg']
                aggr_results = []
                for aggr in self.aggregators:
                    if aggr == 'sum':
                        aggr_results.append(torch.sum(msg, dim=1))
                    elif aggr == 'max':
                        aggr_results.append(torch.max(msg, dim=1)[0])
                    elif aggr == 'min':
                        aggr_results.append(torch.min(msg, dim=1)[0])
                    elif aggr == 'mean':
                        aggr_results.append(torch.mean(msg, dim=1))
                    elif aggr == 'std':
                        var = torch.var(msg, dim=1, unbiased=False)
                        aggr_results.append(torch.sqrt(var + 1e-5))
                return {'m_aggr': torch.cat(aggr_results, dim=-1)}

            g.update_all(message_func, reduce_func)
            m_aggr = g.ndata['m_aggr']

            if self.scalers == ['identity']:
                m_scaled = m_aggr
            else:
                deg = g.in_degrees().float().unsqueeze(-1).clamp(min=1)

                scale_results = []
                for scaler in self.scalers:
                    if scaler == 'identity':
                        scale_results.append(m_aggr)
                    elif scaler == 'amplification':
                        scale_results.append(m_aggr * (torch.log(deg + 1.0) / self.avg_log_d))
                    elif scaler == 'attenuation':
                        scale_results.append(m_aggr * (self.avg_log_d / torch.log(deg + 1.0)))
                m_scaled = torch.cat(scale_results, dim=-1)

            out = self.posttrans(torch.cat([h, m_scaled], dim=-1))

            if self.use_graph_norm and snorm_n is not None:
                out = out * snorm_n

            if self.use_batch_norm:
                out = self.batch_norm(out)

            out = self.dropout(out)

            return out

In [ ]:
class PNALayer(nn.Module):
    def __init__(self, in_features, out_features, aggregators, scalers, avg_log_d, towers=5,
                 pretrans_layers=1, posttrans_layers=1, divide_input=True, residual=True,
                 edge_dim=0, use_batch_norm=True, use_graph_norm=True, dropout=0.0):
        super().__init__()
        self.towers_list = nn.ModuleList()
        self.divide_input = divide_input
        self.residual = residual

        self.input_tower = in_features // towers if divide_input else in_features
        self.output_tower = out_features // towers

        for _ in range(towers):
            self.towers_list.append(
                PNATower(self.input_tower, self.output_tower, aggregators, scalers, avg_log_d,
                         pretrans_layers, posttrans_layers, edge_dim,
                         use_batch_norm, use_graph_norm, dropout)
            )

        self.mixing_network = FCLayer(out_features, out_features, activation='LeakyReLU')

    def forward(self, g, h, snorm_n, e=None):
        h_in = h
        if self.divide_input:
            tower_outs = []
            for i, tower in enumerate(self.towers_list):
                h_chunk = h[:, i * self.input_tower : (i + 1) * self.input_tower]
                tower_outs.append(tower(g, h_chunk, snorm_n, e))
            y = torch.cat(tower_outs, dim=-1)
        else:
            y = torch.cat([tower(g, h, snorm_n, e) for tower in self.towers_list], dim=-1)

        y = self.mixing_network(y)

        if self.residual and h_in.shape == y.shape:
            y = y + h_in

        return y

In [ ]:
class RealWorldBenchmarkModel(nn.Module):
    def __init__(self, config, avg_log_d=None):
        super().__init__()
        self.edge_feat = config.edge_feat

        self.in_feat_dropout = nn.Dropout(config.in_feat_dropout)


        if config.name == 'ZINC':
            self.node_encoder = nn.Embedding(num_embeddings=28, embedding_dim=config.hidden_dim)
            if self.edge_feat:
                self.edge_encoder = nn.Embedding(num_embeddings=4, embedding_dim=config.edge_dim)
                pass_edge_dim = config.edge_dim
            else:
                pass_edge_dim = 0
        elif config.name == 'MolHIV':
            self.node_encoder = AtomEncoder(emb_dim=config.hidden_dim)
            if self.edge_feat:
                self.edge_encoder = BondEncoder(emb_dim=config.edge_dim)
                pass_edge_dim = config.edge_dim
            else:
                pass_edge_dim = 0
        else:
            self.node_encoder = nn.Linear(config.node_dim, config.hidden_dim)
            if self.edge_feat:
                self.edge_encoder = nn.Linear(1, config.edge_dim)
                pass_edge_dim = config.edge_dim
            else:
                pass_edge_dim = 0

        self.layers = nn.ModuleList()


        for i in range(config.num_layers):
            is_first = (i == 0)
            is_last = (i == config.num_layers - 1)

            if is_first: divide_in = config.divide_input_first
            elif is_last: divide_in = config.divide_input_last
            else: divide_in = True

            layer_in_dim = config.hidden_dim
            layer_out_dim = config.out_dim if is_last else config.hidden_dim

            self.layers.append(
                PNALayer(layer_in_dim, layer_out_dim, config.aggregators, config.scalers, avg_log_d,
                         towers=config.towers, pretrans_layers=config.pretrans_layers,
                         posttrans_layers=config.posttrans_layers, divide_input=divide_in,
                         residual=True, edge_dim=pass_edge_dim,
                         use_batch_norm=config.use_batch_norm, use_graph_norm=config.use_graph_norm,
                         dropout=config.dropout)
            )


        self.graph_decoder = nn.Sequential(
            nn.Linear(config.out_dim, config.out_dim // 2),
            nn.ReLU(),
            nn.Linear(config.out_dim // 2, config.task_dim)
        )

    def forward(self, g, h, snorm_n, e=None):

        h = self.node_encoder(h)
        h = self.in_feat_dropout(h)


        if self.edge_feat and e is not None:
            e = self.edge_encoder(e)


        for layer in self.layers:
            h = layer(g, h, snorm_n, e)


        g.ndata['h_out'] = h
        h_graph = dgl.readout_nodes(g, 'h_out', op='sum')


        out = self.graph_decoder(h_graph)
        return out

In [ ]:
def train_epoch(model, optimizer, dataloader, device, config):
    model.train()
    epoch_loss = 0.0
    epoch_metric = 0.0

    for batched_graph, labels in dataloader:
        batched_graph, h, e, labels = config.preprocess(
            batched_graph, labels, device, config.edge_feat
        )

        tab_sizes = batched_graph.batch_num_nodes()
        snorm_n = [torch.ones(size, 1) / math.sqrt(size) for size in tab_sizes]
        snorm_n = torch.cat(snorm_n).to(device)

        pred = model(batched_graph, h, snorm_n, e)

        loss = config.loss_fn(pred, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_size = batched_graph.batch_size
        epoch_loss += loss.item() * batch_size
        epoch_metric += config.calc_metric(pred, labels, batch_size)

    num_samples = len(dataloader.dataset)
    return epoch_loss / num_samples, epoch_metric / num_samples


def evaluate_epoch(model, dataloader, device, config):

    model.eval()
    epoch_loss = 0.0
    epoch_metric = 0.0

    with torch.no_grad():
        for batched_graph, labels in dataloader:
            batched_graph, h, e, labels = config.preprocess(
                batched_graph, labels, device, config.edge_feat
            )

            tab_sizes = batched_graph.batch_num_nodes()
            snorm_n = [torch.ones(size, 1) / math.sqrt(size) for size in tab_sizes]
            snorm_n = torch.cat(snorm_n).to(device)

            pred = model(batched_graph, h, snorm_n, e)

            loss = config.loss_fn(pred, labels)

            batch_size = batched_graph.batch_size
            epoch_loss += loss.item() * batch_size
            epoch_metric += config.calc_metric(pred, labels, batch_size)

    num_samples = len(dataloader.dataset)
    return epoch_loss / num_samples, epoch_metric / num_samples

In [ ]:
class BaseConfig:
    epochs = 1000
    batch_size = 128
    init_lr = 0.001
    lr_reduce_factor = 0.5
    min_lr = 1e-5
    weight_decay = 3e-6

    num_layers = 4
    towers = 5
    pretrans_layers = 1
    posttrans_layers = 1
    use_batch_norm = True
    use_graph_norm = True
    in_feat_dropout = 0.0


# 1. ZINC
class BaseZINCConfig(BaseConfig):
    name = 'ZINC'
    task_dim = 1
    node_dim = 28
    scheduler_mode = 'min'

# --- 1.1 ZINC PNA ---
class ZINC_PNA_NOEDGE(BaseZINCConfig):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity', 'amplification', 'attenuation']
    edge_feat = False
    edge_dim = 0
    hidden_dim = 75
    out_dim = 70
    dropout = 0.0
    divide_input_first = False
    divide_input_last = True
    lr_schedule_patience = 5

class ZINC_PNA_EDGE(BaseZINCConfig):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity', 'amplification', 'attenuation']
    edge_feat = True
    edge_dim = 50
    hidden_dim = 70
    out_dim = 60
    dropout = 0.0
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 20

# --- 1.2 ZINC PNA (No Scalers) ---
class ZINC_PNA_NOSCALER_NOEDGE(BaseZINCConfig):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity']
    edge_feat = False
    edge_dim = 0
    hidden_dim = 95
    out_dim = 90
    dropout = 0.0
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5

class ZINC_PNA_NOSCALER_EDGE(BaseZINCConfig):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity']
    edge_feat = True
    edge_dim = 50
    hidden_dim = 90
    out_dim = 80
    dropout = 0.0
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 20

# --- 1.3 ZINC MPNN (Sum) ---
class ZINC_MPNN_SUM_NOEDGE(BaseZINCConfig):
    aggregators = ['sum']
    scalers = ['identity']
    edge_feat = False
    edge_dim = 0
    hidden_dim = 110
    out_dim = 80
    dropout = 0.0
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5
    weight_decay = 1e-5

class ZINC_MPNN_SUM_EDGE(BaseZINCConfig):
    aggregators = ['sum']
    scalers = ['identity']
    edge_feat = True
    edge_dim = 50
    hidden_dim = 100
    out_dim = 70
    dropout = 0.0
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 20

# --- 1.4 ZINC MPNN (Max) ---
class ZINC_MPNN_MAX_NOEDGE(BaseZINCConfig):
    aggregators = ['max']
    scalers = ['identity']
    edge_feat = False
    edge_dim = 0
    hidden_dim = 110
    out_dim = 80
    dropout = 0.0
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5
    weight_decay = 1e-5

class ZINC_MPNN_MAX_EDGE(BaseZINCConfig):
    aggregators = ['max']
    scalers = ['identity']
    edge_feat = True
    edge_dim = 50
    hidden_dim = 100
    out_dim = 70
    dropout = 0.0
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 20

def preprocess_zinc(batched_graph, labels, device, use_edge_feat=True):
    batched_graph = batched_graph.to(device)
    labels = labels.to(device).float()
    h = batched_graph.ndata['feat'].long().squeeze()

    if use_edge_feat and batched_graph.num_edges() > 0:
        e = batched_graph.edata['feat'].long().squeeze()
    else:
        e = None
    return batched_graph, h, e, labels

def zinc_loss(pred, labels):
    return F.l1_loss(pred.squeeze(), labels.squeeze())

def zinc_metric(pred, labels, batch_size):
    return F.l1_loss(pred.squeeze(), labels.squeeze()).item() * batch_size

In [ ]:
from dgl.data import ZINCDataset

zinc_train = ZINCDataset(mode='train')
zinc_val = ZINCDataset(mode='valid')
zinc_test = ZINCDataset(mode='test')

train_loader = GraphDataLoader(zinc_train, batch_size=128, shuffle=True, drop_last=True, num_workers=2, pin_memory=True)
val_loader = GraphDataLoader(zinc_val, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
test_loader = GraphDataLoader(zinc_test, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

avg_log_d = compute_avg_degree(zinc_train)

In [ ]:
config = ZINC_PNA_EDGE()
config.preprocess = preprocess_zinc
config.loss_fn = zinc_loss
config.calc_metric = zinc_metric

model = RealWorldBenchmarkModel(config, avg_log_d).to(device)

optimizer = optim.Adam(model.parameters(), lr=config.init_lr, weight_decay=config.weight_decay)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=config.scheduler_mode,
    factor=config.lr_reduce_factor,
    patience=config.lr_schedule_patience,
)

best_val_metric = float('inf')
test_metric_at_best_val = 0.0

for epoch in range(config.epochs):
    train_loss, train_metric = train_epoch(model, optimizer, train_loader, device, config)
    val_loss, val_metric = evaluate_epoch(model, val_loader, device, config)

    scheduler.step(val_metric)
    current_lr = optimizer.param_groups[0]['lr']

    if val_metric < best_val_metric:
        best_val_metric = val_metric
        _, test_metric = evaluate_epoch(model, test_loader, device, config)
        test_metric_at_best_val = test_metric
        torch.save(model.state_dict(), 'best_pna_model.pth')
        best_mark = " (New Best!)"
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train: {train_loss:.4f} | Val: {val_metric:.4f} | Test: {test_metric:.4f} {best_mark}")
    else:
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train: {train_loss:.4f} | Val: {val_metric:.4f}")

    if current_lr < config.min_lr:
        break


print(f"🏆  for Best Val : Test MAE = {test_metric_at_best_val:.4f}")


Epoch 000 | LR: 0.001000 | Train: 0.6832 | Val: 0.5809 | Test: 0.6217  (New Best!)
Epoch 001 | LR: 0.001000 | Train: 0.5122 | Val: 0.5241 | Test: 0.5552  (New Best!)
Epoch 002 | LR: 0.001000 | Train: 0.4690 | Val: 0.6152
Epoch 003 | LR: 0.001000 | Train: 0.4161 | Val: 0.6908
Epoch 004 | LR: 0.001000 | Train: 0.3987 | Val: 0.3898 | Test: 0.4486  (New Best!)
Epoch 005 | LR: 0.001000 | Train: 0.3839 | Val: 0.4436
Epoch 006 | LR: 0.001000 | Train: 0.4022 | Val: 0.4655
Epoch 007 | LR: 0.001000 | Train: 0.3945 | Val: 0.8239
Epoch 008 | LR: 0.001000 | Train: 0.3757 | Val: 0.4076
Epoch 009 | LR: 0.001000 | Train: 0.3724 | Val: 0.3470 | Test: 0.3919  (New Best!)
Epoch 010 | LR: 0.001000 | Train: 0.3703 | Val: 0.6436
Epoch 011 | LR: 0.001000 | Train: 0.3779 | Val: 0.6470
Epoch 012 | LR: 0.001000 | Train: 0.3660 | Val: 0.4704
Epoch 013 | LR: 0.001000 | Train: 0.3355 | Val: 0.3897
Epoch 014 | LR: 0.001000 | Train: 0.3789 | Val: 0.3948
Epoch 015 | LR: 0.001000 | Train: 0.3542 | Val: 0.4140
Epoch 01

In [ ]:
config = ZINC_PNA_NOEDGE()
config.preprocess = preprocess_zinc
config.loss_fn = zinc_loss
config.calc_metric = zinc_metric

model = RealWorldBenchmarkModel(config, avg_log_d).to(device)

optimizer = optim.Adam(model.parameters(), lr=config.init_lr, weight_decay=config.weight_decay)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=config.scheduler_mode,
    factor=config.lr_reduce_factor,
    patience=config.lr_schedule_patience,
)

best_val_metric = float('inf')
test_metric_at_best_val = 0.0

for epoch in range(config.epochs):
    train_loss, train_metric = train_epoch(model, optimizer, train_loader, device, config)
    val_loss, val_metric = evaluate_epoch(model, val_loader, device, config)

    scheduler.step(val_metric)
    current_lr = optimizer.param_groups[0]['lr']

    if val_metric < best_val_metric:
        best_val_metric = val_metric
        _, test_metric = evaluate_epoch(model, test_loader, device, config)
        test_metric_at_best_val = test_metric
        torch.save(model.state_dict(), 'best_pna_noedge_model.pth')
        best_mark = " (New Best!)"
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train: {train_loss:.4f} | Val: {val_metric:.4f} | Test: {test_metric:.4f} {best_mark}")
    else:
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train: {train_loss:.4f} | Val: {val_metric:.4f}")

    if current_lr < config.min_lr:
        break


print(f"🏆  for Best Val : Test MAE = {test_metric_at_best_val:.4f}")


Epoch 000 | LR: 0.001000 | Train: 0.6984 | Val: 0.6430 | Test: 0.6901  (New Best!)
Epoch 001 | LR: 0.001000 | Train: 0.5367 | Val: 0.4983 | Test: 0.5289  (New Best!)
Epoch 002 | LR: 0.001000 | Train: 0.5014 | Val: 0.4802 | Test: 0.5082  (New Best!)
Epoch 003 | LR: 0.001000 | Train: 0.5371 | Val: 0.6323
Epoch 004 | LR: 0.001000 | Train: 0.4711 | Val: 0.4432 | Test: 0.4831  (New Best!)
Epoch 005 | LR: 0.001000 | Train: 0.4794 | Val: 0.6307
Epoch 006 | LR: 0.001000 | Train: 0.4472 | Val: 0.5001
Epoch 007 | LR: 0.001000 | Train: 0.4370 | Val: 0.4389 | Test: 0.4590  (New Best!)
Epoch 008 | LR: 0.001000 | Train: 0.4420 | Val: 0.4655
Epoch 009 | LR: 0.001000 | Train: 0.4210 | Val: 0.4535
Epoch 010 | LR: 0.001000 | Train: 0.4287 | Val: 0.6547
Epoch 011 | LR: 0.001000 | Train: 0.4114 | Val: 0.4292 | Test: 0.4522  (New Best!)
Epoch 012 | LR: 0.001000 | Train: 0.4197 | Val: 0.4407
Epoch 013 | LR: 0.001000 | Train: 0.3959 | Val: 0.4513
Epoch 014 | LR: 0.001000 | Train: 0.4186 | Val: 0.4790
Epoch 0

In [ ]:
config = ZINC_PNA_NOSCALER_EDGE()
config.preprocess = preprocess_zinc
config.loss_fn = zinc_loss
config.calc_metric = zinc_metric

model = RealWorldBenchmarkModel(config, avg_log_d).to(device)

optimizer = optim.Adam(model.parameters(), lr=config.init_lr, weight_decay=config.weight_decay)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=config.scheduler_mode,
    factor=config.lr_reduce_factor,
    patience=config.lr_schedule_patience,
)

best_val_metric = float('inf')
test_metric_at_best_val = 0.0

for epoch in range(config.epochs):
    train_loss, train_metric = train_epoch(model, optimizer, train_loader, device, config)
    val_loss, val_metric = evaluate_epoch(model, val_loader, device, config)

    scheduler.step(val_metric)
    current_lr = optimizer.param_groups[0]['lr']

    if val_metric < best_val_metric:
        best_val_metric = val_metric
        _, test_metric = evaluate_epoch(model, test_loader, device, config)
        test_metric_at_best_val = test_metric
        torch.save(model.state_dict(), 'best_pna_noscalers_model.pth')
        best_mark = " (New Best!)"
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train: {train_loss:.4f} | Val: {val_metric:.4f} | Test: {test_metric:.4f} {best_mark}")
    else:
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train: {train_loss:.4f} | Val: {val_metric:.4f}")

    if current_lr < config.min_lr:
        break


print(f"🏆  for Best Val : Test MAE = {test_metric_at_best_val:.4f}")


Epoch 000 | LR: 0.001000 | Train: 0.6886 | Val: 0.5323 | Test: 0.5768  (New Best!)
Epoch 001 | LR: 0.001000 | Train: 0.5422 | Val: 0.5318 | Test: 0.5777  (New Best!)
Epoch 002 | LR: 0.001000 | Train: 0.4909 | Val: 0.6403
Epoch 003 | LR: 0.001000 | Train: 0.4803 | Val: 0.4939 | Test: 0.5351  (New Best!)
Epoch 004 | LR: 0.001000 | Train: 0.4725 | Val: 0.4357 | Test: 0.4874  (New Best!)
Epoch 005 | LR: 0.001000 | Train: 0.4738 | Val: 0.4735
Epoch 006 | LR: 0.001000 | Train: 0.4481 | Val: 0.4478
Epoch 007 | LR: 0.001000 | Train: 0.4186 | Val: 0.4691
Epoch 008 | LR: 0.001000 | Train: 0.4298 | Val: 0.7326
Epoch 009 | LR: 0.001000 | Train: 0.4045 | Val: 0.4400
Epoch 010 | LR: 0.001000 | Train: 0.4078 | Val: 0.4257 | Test: 0.4622  (New Best!)
Epoch 011 | LR: 0.001000 | Train: 0.4246 | Val: 0.4436
Epoch 012 | LR: 0.001000 | Train: 0.3994 | Val: 0.3817 | Test: 0.4335  (New Best!)
Epoch 013 | LR: 0.001000 | Train: 0.3810 | Val: 0.5869
Epoch 014 | LR: 0.001000 | Train: 0.3755 | Val: 0.6484
Epoch 0

In [ ]:
config = ZINC_PNA_NOSCALER_NOEDGE()
config.preprocess = preprocess_zinc
config.loss_fn = zinc_loss
config.calc_metric = zinc_metric

model = RealWorldBenchmarkModel(config, avg_log_d).to(device)

optimizer = optim.Adam(model.parameters(), lr=config.init_lr, weight_decay=config.weight_decay)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=config.scheduler_mode,
    factor=config.lr_reduce_factor,
    patience=config.lr_schedule_patience,
)

best_val_metric = float('inf')
test_metric_at_best_val = 0.0

for epoch in range(config.epochs):
    train_loss, train_metric = train_epoch(model, optimizer, train_loader, device, config)
    val_loss, val_metric = evaluate_epoch(model, val_loader, device, config)

    scheduler.step(val_metric)
    current_lr = optimizer.param_groups[0]['lr']

    if val_metric < best_val_metric:
        best_val_metric = val_metric
        _, test_metric = evaluate_epoch(model, test_loader, device, config)
        test_metric_at_best_val = test_metric
        torch.save(model.state_dict(), 'best_pna_noscalers_noedge_model.pth')
        best_mark = " (New Best!)"
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train: {train_loss:.4f} | Val: {val_metric:.4f} | Test: {test_metric:.4f} {best_mark}")
    else:
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train: {train_loss:.4f} | Val: {val_metric:.4f}")

    if current_lr < config.min_lr:
        break


print(f"🏆  for Best Val : Test MAE = {test_metric_at_best_val:.4f}")


Epoch 000 | LR: 0.001000 | Train: 0.7129 | Val: 0.7909 | Test: 0.8237  (New Best!)
Epoch 001 | LR: 0.001000 | Train: 0.5911 | Val: 0.5435 | Test: 0.5718  (New Best!)
Epoch 002 | LR: 0.001000 | Train: 0.5546 | Val: 0.5947
Epoch 003 | LR: 0.001000 | Train: 0.5792 | Val: 0.5676
Epoch 004 | LR: 0.001000 | Train: 0.5487 | Val: 0.5386 | Test: 0.5616  (New Best!)
Epoch 005 | LR: 0.001000 | Train: 0.5251 | Val: 0.5424
Epoch 006 | LR: 0.001000 | Train: 0.5123 | Val: 0.5360 | Test: 0.5661  (New Best!)
Epoch 007 | LR: 0.001000 | Train: 0.5082 | Val: 0.7919
Epoch 008 | LR: 0.001000 | Train: 0.5149 | Val: 0.5073 | Test: 0.5378  (New Best!)
Epoch 009 | LR: 0.001000 | Train: 0.4954 | Val: 0.4812 | Test: 0.5115  (New Best!)
Epoch 010 | LR: 0.001000 | Train: 0.5012 | Val: 0.4783 | Test: 0.5124  (New Best!)
Epoch 011 | LR: 0.001000 | Train: 0.4718 | Val: 0.4910
Epoch 012 | LR: 0.001000 | Train: 0.4706 | Val: 0.5201
Epoch 013 | LR: 0.001000 | Train: 0.4546 | Val: 0.5691
Epoch 014 | LR: 0.001000 | Train: 

In [ ]:
config = ZINC_MPNN_SUM_EDGE()
config.preprocess = preprocess_zinc
config.loss_fn = zinc_loss
config.calc_metric = zinc_metric

model = RealWorldBenchmarkModel(config, avg_log_d).to(device)

optimizer = optim.Adam(model.parameters(), lr=config.init_lr, weight_decay=config.weight_decay)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=config.scheduler_mode,
    factor=config.lr_reduce_factor,
    patience=config.lr_schedule_patience,
)

best_val_metric = float('inf')
test_metric_at_best_val = 0.0

for epoch in range(config.epochs):
    train_loss, train_metric = train_epoch(model, optimizer, train_loader, device, config)
    val_loss, val_metric = evaluate_epoch(model, val_loader, device, config)

    scheduler.step(val_metric)
    current_lr = optimizer.param_groups[0]['lr']

    if val_metric < best_val_metric:
        best_val_metric = val_metric
        _, test_metric = evaluate_epoch(model, test_loader, device, config)
        test_metric_at_best_val = test_metric
        torch.save(model.state_dict(), 'best_mpnn_sum_model.pth')
        best_mark = " (New Best!)"
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train: {train_loss:.4f} | Val: {val_metric:.4f} | Test: {test_metric:.4f} {best_mark}")
    else:
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train: {train_loss:.4f} | Val: {val_metric:.4f}")

    if current_lr < config.min_lr:
        break


print(f"🏆  for Best Val : Test MAE = {test_metric_at_best_val:.4f}")


Epoch 000 | LR: 0.001000 | Train: 0.6679 | Val: 0.5145 | Test: 0.5436  (New Best!)
Epoch 001 | LR: 0.001000 | Train: 0.4908 | Val: 0.8188
Epoch 002 | LR: 0.001000 | Train: 0.4939 | Val: 0.5145 | Test: 0.5204  (New Best!)
Epoch 003 | LR: 0.001000 | Train: 0.4257 | Val: 0.7557
Epoch 004 | LR: 0.001000 | Train: 0.4196 | Val: 0.4051 | Test: 0.4428  (New Best!)
Epoch 005 | LR: 0.001000 | Train: 0.4184 | Val: 0.3912 | Test: 0.4324  (New Best!)
Epoch 006 | LR: 0.001000 | Train: 0.4051 | Val: 0.7461
Epoch 007 | LR: 0.001000 | Train: 0.3894 | Val: 0.4488
Epoch 008 | LR: 0.001000 | Train: 0.4102 | Val: 0.3860 | Test: 0.4217  (New Best!)
Epoch 009 | LR: 0.001000 | Train: 0.3845 | Val: 0.5910
Epoch 010 | LR: 0.001000 | Train: 0.3908 | Val: 0.3904
Epoch 011 | LR: 0.001000 | Train: 0.3845 | Val: 0.5578
Epoch 012 | LR: 0.001000 | Train: 0.3651 | Val: 0.6240
Epoch 013 | LR: 0.001000 | Train: 0.3666 | Val: 0.4542
Epoch 014 | LR: 0.001000 | Train: 0.3769 | Val: 0.4295
Epoch 015 | LR: 0.001000 | Train: 0

In [ ]:
config = ZINC_MPNN_SUM_NOEDGE()
config.preprocess = preprocess_zinc
config.loss_fn = zinc_loss
config.calc_metric = zinc_metric

model = RealWorldBenchmarkModel(config, avg_log_d).to(device)

optimizer = optim.Adam(model.parameters(), lr=config.init_lr, weight_decay=config.weight_decay)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=config.scheduler_mode,
    factor=config.lr_reduce_factor,
    patience=config.lr_schedule_patience,
)

best_val_metric = float('inf')
test_metric_at_best_val = 0.0

for epoch in range(config.epochs):
    train_loss, train_metric = train_epoch(model, optimizer, train_loader, device, config)
    val_loss, val_metric = evaluate_epoch(model, val_loader, device, config)

    scheduler.step(val_metric)
    current_lr = optimizer.param_groups[0]['lr']

    if val_metric < best_val_metric:
        best_val_metric = val_metric
        _, test_metric = evaluate_epoch(model, test_loader, device, config)
        test_metric_at_best_val = test_metric
        torch.save(model.state_dict(), 'best_mpnn_sum_noedge_model.pth')
        best_mark = " (New Best!)"
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train: {train_loss:.4f} | Val: {val_metric:.4f} | Test: {test_metric:.4f} {best_mark}")
    else:
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train: {train_loss:.4f} | Val: {val_metric:.4f}")

    if current_lr < config.min_lr:
        break


print(f"🏆  for Best Val : Test MAE = {test_metric_at_best_val:.4f}")

Epoch 000 | LR: 0.001000 | Train: 0.6982 | Val: 0.5333 | Test: 0.5733  (New Best!)
Epoch 001 | LR: 0.001000 | Train: 0.5292 | Val: 0.4934 | Test: 0.5341  (New Best!)
Epoch 002 | LR: 0.001000 | Train: 0.5078 | Val: 0.5049
Epoch 003 | LR: 0.001000 | Train: 0.4859 | Val: 0.4591 | Test: 0.5017  (New Best!)
Epoch 004 | LR: 0.001000 | Train: 0.4743 | Val: 0.5233
Epoch 005 | LR: 0.001000 | Train: 0.4854 | Val: 0.5959
Epoch 006 | LR: 0.001000 | Train: 0.4642 | Val: 0.5489
Epoch 007 | LR: 0.001000 | Train: 0.4678 | Val: 0.5047
Epoch 008 | LR: 0.001000 | Train: 0.4372 | Val: 0.5486
Epoch 009 | LR: 0.001000 | Train: 0.4401 | Val: 0.4351 | Test: 0.4659  (New Best!)
Epoch 010 | LR: 0.001000 | Train: 0.4329 | Val: 0.4915
Epoch 011 | LR: 0.001000 | Train: 0.4295 | Val: 0.4534
Epoch 012 | LR: 0.001000 | Train: 0.4208 | Val: 0.9863
Epoch 013 | LR: 0.001000 | Train: 0.4313 | Val: 0.4202 | Test: 0.4313  (New Best!)
Epoch 014 | LR: 0.001000 | Train: 0.4260 | Val: 0.5137
Epoch 015 | LR: 0.001000 | Train: 0

In [ ]:
config = ZINC_MPNN_MAX_EDGE()
config.preprocess = preprocess_zinc
config.loss_fn = zinc_loss
config.calc_metric = zinc_metric

model = RealWorldBenchmarkModel(config, avg_log_d).to(device)

optimizer = optim.Adam(model.parameters(), lr=config.init_lr, weight_decay=config.weight_decay)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=config.scheduler_mode,
    factor=config.lr_reduce_factor,
    patience=config.lr_schedule_patience,
)

best_val_metric = float('inf')
test_metric_at_best_val = 0.0

for epoch in range(config.epochs):
    train_loss, train_metric = train_epoch(model, optimizer, train_loader, device, config)
    val_loss, val_metric = evaluate_epoch(model, val_loader, device, config)

    scheduler.step(val_metric)
    current_lr = optimizer.param_groups[0]['lr']

    if val_metric < best_val_metric:
        best_val_metric = val_metric
        _, test_metric = evaluate_epoch(model, test_loader, device, config)
        test_metric_at_best_val = test_metric
        torch.save(model.state_dict(), 'best_mpnn_max_model.pth')
        best_mark = " (New Best!)"
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train: {train_loss:.4f} | Val: {val_metric:.4f} | Test: {test_metric:.4f} {best_mark}")
    else:
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train: {train_loss:.4f} | Val: {val_metric:.4f}")

    if current_lr < config.min_lr:
        break


print(f"🏆  for Best Val : Test MAE = {test_metric_at_best_val:.4f}")

Epoch 000 | LR: 0.001000 | Train: 0.6777 | Val: 0.5866 | Test: 0.6388  (New Best!)
Epoch 001 | LR: 0.001000 | Train: 0.5809 | Val: 0.9708
Epoch 002 | LR: 0.001000 | Train: 0.5453 | Val: 0.7362
Epoch 003 | LR: 0.001000 | Train: 0.4981 | Val: 0.5325 | Test: 0.5716  (New Best!)
Epoch 004 | LR: 0.001000 | Train: 0.4787 | Val: 0.5050 | Test: 0.5495  (New Best!)
Epoch 005 | LR: 0.001000 | Train: 0.4749 | Val: 0.7070
Epoch 006 | LR: 0.001000 | Train: 0.4462 | Val: 0.4722 | Test: 0.5146  (New Best!)
Epoch 007 | LR: 0.001000 | Train: 0.4434 | Val: 0.5283
Epoch 008 | LR: 0.001000 | Train: 0.4299 | Val: 0.4072 | Test: 0.4627  (New Best!)
Epoch 009 | LR: 0.001000 | Train: 0.4112 | Val: 0.3927 | Test: 0.4399  (New Best!)
Epoch 010 | LR: 0.001000 | Train: 0.4266 | Val: 0.4492
Epoch 011 | LR: 0.001000 | Train: 0.3986 | Val: 0.6685
Epoch 012 | LR: 0.001000 | Train: 0.3884 | Val: 0.4014
Epoch 013 | LR: 0.001000 | Train: 0.4003 | Val: 0.6240
Epoch 014 | LR: 0.001000 | Train: 0.3879 | Val: 0.3953
Epoch 0

In [ ]:
config = ZINC_MPNN_MAX_NOEDGE()
config.preprocess = preprocess_zinc
config.loss_fn = zinc_loss
config.calc_metric = zinc_metric

model = RealWorldBenchmarkModel(config, avg_log_d).to(device)

optimizer = optim.Adam(model.parameters(), lr=config.init_lr, weight_decay=config.weight_decay)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=config.scheduler_mode,
    factor=config.lr_reduce_factor,
    patience=config.lr_schedule_patience,
)

best_val_metric = float('inf')
test_metric_at_best_val = 0.0

for epoch in range(config.epochs):
    train_loss, train_metric = train_epoch(model, optimizer, train_loader, device, config)
    val_loss, val_metric = evaluate_epoch(model, val_loader, device, config)

    scheduler.step(val_metric)
    current_lr = optimizer.param_groups[0]['lr']

    if val_metric < best_val_metric:
        best_val_metric = val_metric
        _, test_metric = evaluate_epoch(model, test_loader, device, config)
        test_metric_at_best_val = test_metric
        torch.save(model.state_dict(), 'best_mpnn_max_noedge_model.pth')
        best_mark = " (New Best!)"
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train: {train_loss:.4f} | Val: {val_metric:.4f} | Test: {test_metric:.4f} {best_mark}")
    else:
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train: {train_loss:.4f} | Val: {val_metric:.4f}")

    if current_lr < config.min_lr:
        break


print(f"🏆  for Best Val : Test MAE = {test_metric_at_best_val:.4f}")

Epoch 000 | LR: 0.001000 | Train: 0.7757 | Val: 0.6302 | Test: 0.6783  (New Best!)
Epoch 001 | LR: 0.001000 | Train: 0.5900 | Val: 0.5839 | Test: 0.6206  (New Best!)
Epoch 002 | LR: 0.001000 | Train: 0.5754 | Val: 0.6332
Epoch 003 | LR: 0.001000 | Train: 0.5576 | Val: 0.5545 | Test: 0.5880  (New Best!)
Epoch 004 | LR: 0.001000 | Train: 0.5639 | Val: 0.5422 | Test: 0.5760  (New Best!)
Epoch 005 | LR: 0.001000 | Train: 0.5496 | Val: 0.5528
Epoch 006 | LR: 0.001000 | Train: 0.5424 | Val: 0.5335 | Test: 0.5659  (New Best!)
Epoch 007 | LR: 0.001000 | Train: 0.5356 | Val: 0.7539
Epoch 008 | LR: 0.001000 | Train: 0.5529 | Val: 0.5551
Epoch 009 | LR: 0.001000 | Train: 0.5398 | Val: 0.5819
Epoch 010 | LR: 0.001000 | Train: 0.5338 | Val: 0.5131 | Test: 0.5547  (New Best!)
Epoch 011 | LR: 0.001000 | Train: 0.5239 | Val: 0.6932
Epoch 012 | LR: 0.001000 | Train: 0.5163 | Val: 0.5902
Epoch 013 | LR: 0.001000 | Train: 0.5242 | Val: 0.5485
Epoch 014 | LR: 0.001000 | Train: 0.4990 | Val: 0.5286
Epoch 0

In [ ]:
class BaseCIFAR10Config(BaseConfig):
    name = 'CIFAR10'
    task_dim = 10
    node_dim = 5
    scheduler_mode = 'max'

# --- 2.1 CIFAR10 PNA ---
class CIFAR10_PNA_NOEDGE(BaseCIFAR10Config):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity', 'amplification', 'attenuation']
    edge_feat = False
    edge_dim = 0
    hidden_dim = 75
    out_dim = 70
    dropout = 0.1
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5

class CIFAR10_PNA_EDGE(BaseCIFAR10Config):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity', 'amplification', 'attenuation']
    edge_feat = True
    edge_dim = 50
    hidden_dim = 75
    out_dim = 70
    dropout = 0.3
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5

# --- 2.2 CIFAR10 PNA (No Scalers) ---
class CIFAR10_PNA_NOSCALER_NOEDGE(BaseCIFAR10Config):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity']
    edge_feat = False
    edge_dim = 0
    hidden_dim = 95
    out_dim = 90
    dropout = 0.1
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5

class CIFAR10_PNA_NOSCALER_EDGE(BaseCIFAR10Config):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity']
    edge_feat = True
    edge_dim = 50
    hidden_dim = 95
    out_dim = 90
    dropout = 0.3
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5

# --- 2.3 CIFAR10 MPNN (Sum) ---
class CIFAR10_MPNN_SUM_NOEDGE(BaseCIFAR10Config):
    aggregators = ['sum']
    scalers = ['identity']
    edge_feat = False
    edge_dim = 0
    hidden_dim = 110
    out_dim = 90
    dropout = 0.2
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5
    weight_decay = 3e-6

class CIFAR10_MPNN_SUM_EDGE(BaseCIFAR10Config):
    aggregators = ['sum']
    scalers = ['identity']
    edge_feat = True
    edge_dim = 20
    hidden_dim = 110
    out_dim = 90
    dropout = 0.2
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5

# --- 2.4 CIFAR10 MPNN (Max) ---
class CIFAR10_MPNN_MAX_NOEDGE(BaseCIFAR10Config):
    aggregators = ['max']
    scalers = ['identity']
    edge_feat = False
    edge_dim = 0
    hidden_dim = 110
    out_dim = 90
    dropout = 0.2
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5
    weight_decay = 3e-6

class CIFAR10_MPNN_MAX_EDGE(BaseCIFAR10Config):
    aggregators = ['max']
    scalers = ['identity']
    edge_feat = True
    edge_dim = 20
    hidden_dim = 110
    out_dim = 90
    dropout = 0.2
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5



class BaseMNISTConfig(BaseConfig):
    name = 'MNIST'
    task_dim = 10
    node_dim = 3
    scheduler_mode = 'max'

# --- 3.1 MNIST PNA ---
class MNIST_PNA_NOEDGE(BaseMNISTConfig):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity', 'amplification', 'attenuation']
    edge_feat = False
    edge_dim = 0
    hidden_dim = 75
    out_dim = 70
    dropout = 0.1
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5

class MNIST_PNA_EDGE(BaseMNISTConfig):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity', 'amplification', 'attenuation']
    edge_feat = True
    edge_dim = 50
    hidden_dim = 75
    out_dim = 70
    dropout = 0.3
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5

# --- 3.2 MNIST PNA (No Scalers) ---
class MNIST_PNA_NOSCALER_NOEDGE(BaseMNISTConfig):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity']
    edge_feat = False
    edge_dim = 0
    hidden_dim = 95
    out_dim = 90
    dropout = 0.1
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5

class MNIST_PNA_NOSCALER_EDGE(BaseMNISTConfig):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity']
    edge_feat = True
    edge_dim = 50
    hidden_dim = 95
    out_dim = 90
    dropout = 0.3
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5

# --- 3.3 MNIST MPNN (Sum) ---
class MNIST_MPNN_SUM_NOEDGE(BaseMNISTConfig):
    aggregators = ['sum']
    scalers = ['identity']
    edge_feat = False
    edge_dim = 0
    hidden_dim = 110
    out_dim = 90
    dropout = 0.2
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5
    weight_decay = 3e-6

class MNIST_MPNN_SUM_EDGE(BaseMNISTConfig):
    aggregators = ['sum']
    scalers = ['identity']
    edge_feat = True
    edge_dim = 20
    hidden_dim = 110
    out_dim = 90
    dropout = 0.2
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5

# --- 3.4 MNIST MPNN (Max) ---
class MNIST_MPNN_MAX_NOEDGE(BaseMNISTConfig):
    aggregators = ['max']
    scalers = ['identity']
    edge_feat = False
    edge_dim = 0
    hidden_dim = 110
    out_dim = 90
    dropout = 0.2
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5
    weight_decay = 3e-6

class MNIST_MPNN_MAX_EDGE(BaseMNISTConfig):
    aggregators = ['max']
    scalers = ['identity']
    edge_feat = True
    edge_dim = 20
    hidden_dim = 110
    out_dim = 90
    dropout = 0.2
    divide_input_first = True
    divide_input_last = True
    lr_schedule_patience = 5


def preprocess_image(batched_graph, labels, device, use_edge_feat=True):
    batched_graph = batched_graph.to(device)
    labels = labels.to(device).long()
    h = batched_graph.ndata['feat'].float()
    if use_edge_feat and batched_graph.num_edges() > 0:
        e = batched_graph.edata['feat'].float()
        if e.dim() == 1:
            e = e.unsqueeze(-1)
    else:
        e = None
    return batched_graph, h, e, labels

def image_loss(pred, labels):
    return F.cross_entropy(pred, labels)

def image_metric(pred, labels, batch_size):
    preds = pred.argmax(dim=1)
    correct = (preds == labels).sum().item()
    return correct

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /root/.dgl/Superpixel_c7da4aa2
!unzip -q /content/drive/MyDrive/Superpixel.zip -d /root/.dgl/Superpixel_c7da4aa2/

!mv /root/.dgl/Superpixel_c7da4aa2/Superpixel_c7da4aa2/* /root/.dgl/Superpixel_c7da4aa2/

!rm -rf /root/.dgl/Superpixel_c7da4aa2/__MACOSX
!rm -rf /root/.dgl/Superpixel_c7da4aa2/Superpixel_c7da4aa2


!ls /root/.dgl/Superpixel_c7da4aa2/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
processed  superpixels	superpixels.zip


In [ ]:
from dgl.data import CIFAR10SuperPixelDataset
from torch.utils.data import random_split

cifar_test = CIFAR10SuperPixelDataset(split='test')
full_train_dataset = CIFAR10SuperPixelDataset(split='train')

train_size = 45000
val_size = len(full_train_dataset) - train_size

generator = torch.Generator().manual_seed(42)
cifar_train, cifar_val = random_split(full_train_dataset, [train_size, val_size], generator=generator)


train_loader = GraphDataLoader(cifar_train, batch_size=128, shuffle=True, drop_last=True, num_workers=2, pin_memory=True)
val_loader = GraphDataLoader(cifar_val, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
test_loader = GraphDataLoader(cifar_test, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

avg_log_d = compute_avg_degree(cifar_train)

To save time, I used two Jupyter Notebook files to train the model simultaneously on Google Colab and Kaggle. As a result, this file contains only half of the training results for the CIFAR10 and MNIST training datasets. Since the code is identical, I only submitted this file.

In [ ]:
config = CIFAR10_PNA_NOEDGE()
config.preprocess = preprocess_image
config.loss_fn = image_loss
config.calc_metric = image_metric

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RealWorldBenchmarkModel(config, avg_log_d).to(device)

optimizer = optim.Adam(model.parameters(), lr=config.init_lr, weight_decay=config.weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=config.scheduler_mode,
    factor=config.lr_reduce_factor,
    patience=config.lr_schedule_patience,
)

best_val_metric = 0.0
test_metric_at_best_val = 0.0

for epoch in range(config.epochs):
    train_loss, train_metric = train_epoch(model, optimizer, train_loader, device, config)
    val_loss, val_metric = evaluate_epoch(model, val_loader, device, config)

    scheduler.step(val_metric)
    current_lr = optimizer.param_groups[0]['lr']

    if val_metric > best_val_metric:
        best_val_metric = val_metric
        _, test_metric = evaluate_epoch(model, test_loader, device, config)
        test_metric_at_best_val = test_metric
        torch.save(model.state_dict(), 'best_pna_noedge_cifar10_model.pth')
        best_mark = " (New Best!)"
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Val Acc: {val_metric:.4f} | Test Acc: {test_metric:.4f} {best_mark}")
    else:
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Val Acc: {val_metric:.4f}")

    if current_lr < config.min_lr:
        break

print(f"🏆 for Best Val : Test Accuracy = {test_metric_at_best_val:.4f}")

Epoch 000 | LR: 0.001000 | Train Loss: 1.8513 | Val Acc: 0.3682 | Test Acc: 0.3772  (New Best!)
Epoch 001 | LR: 0.001000 | Train Loss: 1.4924 | Val Acc: 0.3818 | Test Acc: 0.4002  (New Best!)
Epoch 002 | LR: 0.001000 | Train Loss: 1.3487 | Val Acc: 0.4544 | Test Acc: 0.4675  (New Best!)
Epoch 003 | LR: 0.001000 | Train Loss: 1.2597 | Val Acc: 0.4400
Epoch 004 | LR: 0.001000 | Train Loss: 1.1949 | Val Acc: 0.5186 | Test Acc: 0.5235  (New Best!)
Epoch 005 | LR: 0.001000 | Train Loss: 1.1424 | Val Acc: 0.5358 | Test Acc: 0.5356  (New Best!)
Epoch 006 | LR: 0.001000 | Train Loss: 1.1055 | Val Acc: 0.5696 | Test Acc: 0.5633  (New Best!)
Epoch 007 | LR: 0.001000 | Train Loss: 1.0739 | Val Acc: 0.5892 | Test Acc: 0.5904  (New Best!)
Epoch 008 | LR: 0.001000 | Train Loss: 1.0545 | Val Acc: 0.5734
Epoch 009 | LR: 0.001000 | Train Loss: 1.0227 | Val Acc: 0.6180 | Test Acc: 0.6114  (New Best!)
Epoch 010 | LR: 0.001000 | Train Loss: 1.0087 | Val Acc: 0.5938
Epoch 011 | LR: 0.001000 | Train Loss: 0

In [ ]:
config = CIFAR10_PNA_NOSCALER_NOEDGE()
config.preprocess = preprocess_image
config.loss_fn = image_loss
config.calc_metric = image_metric

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RealWorldBenchmarkModel(config, avg_log_d).to(device)

optimizer = optim.Adam(model.parameters(), lr=config.init_lr, weight_decay=config.weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=config.scheduler_mode,
    factor=config.lr_reduce_factor,
    patience=config.lr_schedule_patience,
)

best_val_metric = 0.0
test_metric_at_best_val = 0.0

for epoch in range(config.epochs):
    train_loss, train_metric = train_epoch(model, optimizer, train_loader, device, config)
    val_loss, val_metric = evaluate_epoch(model, val_loader, device, config)

    scheduler.step(val_metric)
    current_lr = optimizer.param_groups[0]['lr']

    if val_metric > best_val_metric:
        best_val_metric = val_metric
        _, test_metric = evaluate_epoch(model, test_loader, device, config)
        test_metric_at_best_val = test_metric
        torch.save(model.state_dict(), 'best_pna_noscaler_noedge_cifar10_model.pth')
        best_mark = " (New Best!)"
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Val Acc: {val_metric:.4f} | Test Acc: {test_metric:.4f} {best_mark}")
    else:
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Val Acc: {val_metric:.4f}")

    if current_lr < config.min_lr:
        break

print(f"🏆 for Best Val : Test Accuracy = {test_metric_at_best_val:.4f}")

Epoch 000 | LR: 0.001000 | Train Loss: 1.8331 | Val Acc: 0.3236 | Test Acc: 0.3332  (New Best!)
Epoch 001 | LR: 0.001000 | Train Loss: 1.4454 | Val Acc: 0.3644 | Test Acc: 0.3778  (New Best!)
Epoch 002 | LR: 0.001000 | Train Loss: 1.3123 | Val Acc: 0.5430 | Test Acc: 0.5385  (New Best!)
Epoch 003 | LR: 0.001000 | Train Loss: 1.2383 | Val Acc: 0.5346
Epoch 004 | LR: 0.001000 | Train Loss: 1.1798 | Val Acc: 0.5478 | Test Acc: 0.5326  (New Best!)
Epoch 005 | LR: 0.001000 | Train Loss: 1.1338 | Val Acc: 0.5476
Epoch 006 | LR: 0.001000 | Train Loss: 1.0959 | Val Acc: 0.5722 | Test Acc: 0.5767  (New Best!)
Epoch 007 | LR: 0.001000 | Train Loss: 1.0621 | Val Acc: 0.5590
Epoch 008 | LR: 0.001000 | Train Loss: 1.0374 | Val Acc: 0.6122 | Test Acc: 0.6071  (New Best!)
Epoch 009 | LR: 0.001000 | Train Loss: 1.0075 | Val Acc: 0.5756
Epoch 010 | LR: 0.001000 | Train Loss: 0.9848 | Val Acc: 0.5094
Epoch 011 | LR: 0.001000 | Train Loss: 0.9612 | Val Acc: 0.6258 | Test Acc: 0.6261  (New Best!)
Epoch 01

In [ ]:
config = CIFAR10_MPNN_SUM_NOEDGE()
config.epochs = 40
config.preprocess = preprocess_image
config.loss_fn = image_loss
config.calc_metric = image_metric

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RealWorldBenchmarkModel(config, avg_log_d).to(device)

optimizer = optim.Adam(model.parameters(), lr=config.init_lr, weight_decay=config.weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=config.scheduler_mode,
    factor=config.lr_reduce_factor,
    patience=config.lr_schedule_patience,
)

best_val_metric = 0.0
test_metric_at_best_val = 0.0

for epoch in range(config.epochs):
    train_loss, train_metric = train_epoch(model, optimizer, train_loader, device, config)
    val_loss, val_metric = evaluate_epoch(model, val_loader, device, config)

    scheduler.step(val_metric)
    current_lr = optimizer.param_groups[0]['lr']

    if val_metric > best_val_metric:
        best_val_metric = val_metric
        _, test_metric = evaluate_epoch(model, test_loader, device, config)
        test_metric_at_best_val = test_metric
        torch.save(model.state_dict(), 'best_mpnn_sum_noedge_cifar10_model.pth')
        best_mark = " (New Best!)"
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Val Acc: {val_metric:.4f} | Test Acc: {test_metric:.4f} {best_mark}")
    else:
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Val Acc: {val_metric:.4f}")

    if current_lr < config.min_lr:
        break

print(f"🏆 for Best Val : Test Accuracy = {test_metric_at_best_val:.4f}")

Epoch 000 | LR: 0.001000 | Train Loss: 1.9672 | Val Acc: 0.3468 | Test Acc: 0.3538  (New Best!)
Epoch 001 | LR: 0.001000 | Train Loss: 1.6365 | Val Acc: 0.4170 | Test Acc: 0.4184  (New Best!)
Epoch 002 | LR: 0.001000 | Train Loss: 1.5124 | Val Acc: 0.4352 | Test Acc: 0.4294  (New Best!)
Epoch 003 | LR: 0.001000 | Train Loss: 1.4357 | Val Acc: 0.3962
Epoch 004 | LR: 0.001000 | Train Loss: 1.3859 | Val Acc: 0.4760 | Test Acc: 0.4778  (New Best!)
Epoch 005 | LR: 0.001000 | Train Loss: 1.3529 | Val Acc: 0.4840 | Test Acc: 0.4902  (New Best!)
Epoch 006 | LR: 0.001000 | Train Loss: 1.3263 | Val Acc: 0.4974 | Test Acc: 0.5039  (New Best!)
Epoch 007 | LR: 0.001000 | Train Loss: 1.3013 | Val Acc: 0.5068 | Test Acc: 0.5033  (New Best!)
Epoch 008 | LR: 0.001000 | Train Loss: 1.2818 | Val Acc: 0.4646
Epoch 009 | LR: 0.001000 | Train Loss: 1.2605 | Val Acc: 0.5512 | Test Acc: 0.5507  (New Best!)
Epoch 010 | LR: 0.001000 | Train Loss: 1.2461 | Val Acc: 0.5462
Epoch 011 | LR: 0.001000 | Train Loss: 1

In [ ]:
config = CIFAR10_MPNN_SUM_EDGE()
config.epochs = 40
config.preprocess = preprocess_image10
config.loss_fn = image_loss
config.calc_metric = image_metric

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RealWorldBenchmarkModel(config, avg_log_d).to(device)

optimizer = optim.Adam(model.parameters(), lr=config.init_lr, weight_decay=config.weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=config.scheduler_mode,
    factor=config.lr_reduce_factor,
    patience=config.lr_schedule_patience,
)

best_val_metric = 0.0
test_metric_at_best_val = 0.0

for epoch in range(config.epochs):
    train_loss, train_metric = train_epoch(model, optimizer, train_loader, device, config)
    val_loss, val_metric = evaluate_epoch(model, val_loader, device, config)

    scheduler.step(val_metric)
    current_lr = optimizer.param_groups[0]['lr']

    if val_metric > best_val_metric:
        best_val_metric = val_metric
        _, test_metric = evaluate_epoch(model, test_loader, device, config)
        test_metric_at_best_val = test_metric
        torch.save(model.state_dict(), 'best_mpnn_sum_cifar10_model.pth')
        best_mark = " (New Best!)"
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Val Acc: {val_metric:.4f} | Test Acc: {test_metric:.4f} {best_mark}")
    else:
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Val Acc: {val_metric:.4f}")

    if current_lr < config.min_lr:
        break

print(f"🏆 for Best Val : Test Accuracy = {test_metric_at_best_val:.4f}")

Epoch 000 | LR: 0.001000 | Train Loss: 1.9773 | Val Acc: 0.3048 | Test Acc: 0.3065  (New Best!)
Epoch 001 | LR: 0.001000 | Train Loss: 1.6378 | Val Acc: 0.2416
Epoch 002 | LR: 0.001000 | Train Loss: 1.5199 | Val Acc: 0.3966 | Test Acc: 0.3843  (New Best!)
Epoch 003 | LR: 0.001000 | Train Loss: 1.4493 | Val Acc: 0.4704 | Test Acc: 0.4758  (New Best!)
Epoch 004 | LR: 0.001000 | Train Loss: 1.4063 | Val Acc: 0.4098
Epoch 005 | LR: 0.001000 | Train Loss: 1.3636 | Val Acc: 0.2666
Epoch 006 | LR: 0.001000 | Train Loss: 1.3345 | Val Acc: 0.4976 | Test Acc: 0.5008  (New Best!)
Epoch 007 | LR: 0.001000 | Train Loss: 1.3046 | Val Acc: 0.4120
Epoch 008 | LR: 0.001000 | Train Loss: 1.2848 | Val Acc: 0.3154
Epoch 009 | LR: 0.001000 | Train Loss: 1.2648 | Val Acc: 0.4982 | Test Acc: 0.4995  (New Best!)
Epoch 010 | LR: 0.001000 | Train Loss: 1.2506 | Val Acc: 0.5002 | Test Acc: 0.5095  (New Best!)
Epoch 011 | LR: 0.001000 | Train Loss: 1.2341 | Val Acc: 0.4854
Epoch 012 | LR: 0.001000 | Train Loss: 1

In [ ]:
from dgl.data import MNISTSuperPixelDataset
from torch.utils.data import random_split

mnist_test = MNISTSuperPixelDataset(split='test')
full_train_dataset = MNISTSuperPixelDataset(split='train')

train_size = 55000
val_size = len(full_train_dataset) - train_size

generator = torch.Generator().manual_seed(42)
mnist_train, mnist_val = random_split(full_train_dataset, [train_size, val_size], generator=generator)


train_loader = GraphDataLoader(mnist_train, batch_size=128, shuffle=True, drop_last=True, num_workers=2, pin_memory=True)
val_loader = GraphDataLoader(mnist_val, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
test_loader = GraphDataLoader(mnist_test, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

avg_log_d = compute_avg_degree(mnist_train)

In [ ]:
config = MNIST_PNA_EDGE()
config.epochs = 40
config.preprocess = preprocess_image
config.loss_fn = image_loss
config.calc_metric = image_metric

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RealWorldBenchmarkModel(config, avg_log_d).to(device)

optimizer = optim.Adam(model.parameters(), lr=config.init_lr, weight_decay=config.weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=config.scheduler_mode,
    factor=config.lr_reduce_factor,
    patience=config.lr_schedule_patience,
)


best_val_metric = 0.0
test_metric_at_best_val = 0.0

for epoch in range(config.epochs):
    train_loss, train_metric = train_epoch(model, optimizer, train_loader, device, config)
    val_loss, val_metric = evaluate_epoch(model, val_loader, device, config)

    scheduler.step(val_metric)
    current_lr = optimizer.param_groups[0]['lr']

    if val_metric > best_val_metric:
        best_val_metric = val_metric
        _, test_metric = evaluate_epoch(model, test_loader, device, config)
        test_metric_at_best_val = test_metric
        torch.save(model.state_dict(), 'best_pna_mnist_model.pth')
        best_mark = " (New Best!)"
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Val Acc: {val_metric:.4f} | Test Acc: {test_metric:.4f} {best_mark}")
    else:
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Val Acc: {val_metric:.4f}")

    if current_lr < config.min_lr:
        break

print(f"🏆 for Best Val : Test Accuracy = {test_metric_at_best_val:.4f}")

Epoch 000 | LR: 0.001000 | Train Loss: 1.2770 | Val Acc: 0.4890 | Test Acc: 0.4959  (New Best!)
Epoch 001 | LR: 0.001000 | Train Loss: 0.4756 | Val Acc: 0.6912 | Test Acc: 0.6973  (New Best!)
Epoch 002 | LR: 0.001000 | Train Loss: 0.3458 | Val Acc: 0.7064 | Test Acc: 0.7171  (New Best!)
Epoch 003 | LR: 0.001000 | Train Loss: 0.2827 | Val Acc: 0.7756 | Test Acc: 0.7890  (New Best!)
Epoch 004 | LR: 0.001000 | Train Loss: 0.2499 | Val Acc: 0.8866 | Test Acc: 0.8955  (New Best!)
Epoch 005 | LR: 0.001000 | Train Loss: 0.2268 | Val Acc: 0.8902 | Test Acc: 0.8932  (New Best!)
Epoch 006 | LR: 0.001000 | Train Loss: 0.2106 | Val Acc: 0.8280
Epoch 007 | LR: 0.001000 | Train Loss: 0.1920 | Val Acc: 0.8978 | Test Acc: 0.9088  (New Best!)
Epoch 008 | LR: 0.001000 | Train Loss: 0.1816 | Val Acc: 0.8352
Epoch 009 | LR: 0.001000 | Train Loss: 0.1776 | Val Acc: 0.9126 | Test Acc: 0.9224  (New Best!)
Epoch 010 | LR: 0.001000 | Train Loss: 0.1698 | Val Acc: 0.9420 | Test Acc: 0.9411  (New Best!)
Epoch 01

In [ ]:
config = MNIST_PNA_NOSCALER_NOEDGE()
config.epochs = 40
config.preprocess = preprocess_image
config.loss_fn = image_loss
config.calc_metric = image_metric

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RealWorldBenchmarkModel(config, avg_log_d).to(device)

optimizer = optim.Adam(model.parameters(), lr=config.init_lr, weight_decay=config.weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=config.scheduler_mode,
    factor=config.lr_reduce_factor,
    patience=config.lr_schedule_patience,
)


best_val_metric = 0.0
test_metric_at_best_val = 0.0

for epoch in range(config.epochs):
    train_loss, train_metric = train_epoch(model, optimizer, train_loader, device, config)
    val_loss, val_metric = evaluate_epoch(model, val_loader, device, config)

    scheduler.step(val_metric)
    current_lr = optimizer.param_groups[0]['lr']

    if val_metric > best_val_metric:
        best_val_metric = val_metric
        _, test_metric = evaluate_epoch(model, test_loader, device, config)
        test_metric_at_best_val = test_metric
        torch.save(model.state_dict(), 'best_pna_mnist_noscalers_noedge_model.pth')
        best_mark = " (New Best!)"
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Val Acc: {val_metric:.4f} | Test Acc: {test_metric:.4f} {best_mark}")
    else:
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Val Acc: {val_metric:.4f}")

    if current_lr < config.min_lr:
        break

print(f"🏆 for Best Val : Test Accuracy = {test_metric_at_best_val:.4f}")

Epoch 000 | LR: 0.001000 | Train Loss: 0.8019 | Val Acc: 0.8332 | Test Acc: 0.8403  (New Best!)
Epoch 001 | LR: 0.001000 | Train Loss: 0.3048 | Val Acc: 0.8784 | Test Acc: 0.8844  (New Best!)
Epoch 002 | LR: 0.001000 | Train Loss: 0.2404 | Val Acc: 0.8834 | Test Acc: 0.8935  (New Best!)
Epoch 003 | LR: 0.001000 | Train Loss: 0.2097 | Val Acc: 0.8596
Epoch 004 | LR: 0.001000 | Train Loss: 0.1848 | Val Acc: 0.9256 | Test Acc: 0.9243  (New Best!)
Epoch 005 | LR: 0.001000 | Train Loss: 0.1738 | Val Acc: 0.9432 | Test Acc: 0.9389  (New Best!)
Epoch 006 | LR: 0.001000 | Train Loss: 0.1585 | Val Acc: 0.9426
Epoch 007 | LR: 0.001000 | Train Loss: 0.1519 | Val Acc: 0.9344
Epoch 008 | LR: 0.001000 | Train Loss: 0.1447 | Val Acc: 0.8948
Epoch 009 | LR: 0.001000 | Train Loss: 0.1369 | Val Acc: 0.9460 | Test Acc: 0.9484  (New Best!)
Epoch 010 | LR: 0.001000 | Train Loss: 0.1337 | Val Acc: 0.9510 | Test Acc: 0.9488  (New Best!)
Epoch 011 | LR: 0.001000 | Train Loss: 0.1262 | Val Acc: 0.9374
Epoch 01

In [ ]:
config = MNIST_MPNN_SUM_NOEDGE()
config.epochs = 40
config.preprocess = preprocess_image
config.loss_fn = image_loss
config.calc_metric = image_metric

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RealWorldBenchmarkModel(config, avg_log_d).to(device)

optimizer = optim.Adam(model.parameters(), lr=config.init_lr, weight_decay=config.weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=config.scheduler_mode,
    factor=config.lr_reduce_factor,
    patience=config.lr_schedule_patience,
)


best_val_metric = 0.0
test_metric_at_best_val = 0.0

for epoch in range(config.epochs):
    train_loss, train_metric = train_epoch(model, optimizer, train_loader, device, config)
    val_loss, val_metric = evaluate_epoch(model, val_loader, device, config)

    scheduler.step(val_metric)
    current_lr = optimizer.param_groups[0]['lr']

    if val_metric > best_val_metric:
        best_val_metric = val_metric
        _, test_metric = evaluate_epoch(model, test_loader, device, config)
        test_metric_at_best_val = test_metric
        torch.save(model.state_dict(), 'best_mpnn_mnist_sum_noedge_model.pth')
        best_mark = " (New Best!)"
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Val Acc: {val_metric:.4f} | Test Acc: {test_metric:.4f} {best_mark}")
    else:
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Val Acc: {val_metric:.4f}")

    if current_lr < config.min_lr:
        break

print(f"🏆 for Best Val : Test Accuracy = {test_metric_at_best_val:.4f}")

Epoch 000 | LR: 0.001000 | Train Loss: 1.2003 | Val Acc: 0.6670 | Test Acc: 0.6930  (New Best!)
Epoch 001 | LR: 0.001000 | Train Loss: 0.5856 | Val Acc: 0.6852 | Test Acc: 0.6955  (New Best!)
Epoch 002 | LR: 0.001000 | Train Loss: 0.4528 | Val Acc: 0.8052 | Test Acc: 0.8220  (New Best!)
Epoch 003 | LR: 0.001000 | Train Loss: 0.3912 | Val Acc: 0.8512 | Test Acc: 0.8736  (New Best!)
Epoch 004 | LR: 0.001000 | Train Loss: 0.3468 | Val Acc: 0.8792 | Test Acc: 0.8871  (New Best!)
Epoch 005 | LR: 0.001000 | Train Loss: 0.3237 | Val Acc: 0.8816 | Test Acc: 0.8872  (New Best!)
Epoch 006 | LR: 0.001000 | Train Loss: 0.2989 | Val Acc: 0.9060 | Test Acc: 0.9170  (New Best!)
Epoch 007 | LR: 0.001000 | Train Loss: 0.2785 | Val Acc: 0.9040
Epoch 008 | LR: 0.001000 | Train Loss: 0.2665 | Val Acc: 0.9226 | Test Acc: 0.9266  (New Best!)
Epoch 009 | LR: 0.001000 | Train Loss: 0.2573 | Val Acc: 0.9264 | Test Acc: 0.9314  (New Best!)
Epoch 010 | LR: 0.001000 | Train Loss: 0.2407 | Val Acc: 0.8992
Epoch 01

In [ ]:
config = MNIST_MPNN_SUM_EDGE()
config.epochs = 40
config.preprocess = preprocess_image
config.loss_fn = image_loss
config.calc_metric = image_metric

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RealWorldBenchmarkModel(config, avg_log_d).to(device)

optimizer = optim.Adam(model.parameters(), lr=config.init_lr, weight_decay=config.weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=config.scheduler_mode,
    factor=config.lr_reduce_factor,
    patience=config.lr_schedule_patience,
)


best_val_metric = 0.0
test_metric_at_best_val = 0.0

for epoch in range(config.epochs):
    train_loss, train_metric = train_epoch(model, optimizer, train_loader, device, config)
    val_loss, val_metric = evaluate_epoch(model, val_loader, device, config)

    scheduler.step(val_metric)
    current_lr = optimizer.param_groups[0]['lr']

    if val_metric > best_val_metric:
        best_val_metric = val_metric
        _, test_metric = evaluate_epoch(model, test_loader, device, config)
        test_metric_at_best_val = test_metric
        torch.save(model.state_dict(), 'best_mpnn_mnist_sum_model.pth')
        best_mark = " (New Best!)"
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Val Acc: {val_metric:.4f} | Test Acc: {test_metric:.4f} {best_mark}")
    else:
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Val Acc: {val_metric:.4f}")

    if current_lr < config.min_lr:
        break

print(f"🏆 for Best Val : Test Accuracy = {test_metric_at_best_val:.4f}")

Epoch 000 | LR: 0.001000 | Train Loss: 1.1410 | Val Acc: 0.3330 | Test Acc: 0.3370  (New Best!)
Epoch 001 | LR: 0.001000 | Train Loss: 0.5448 | Val Acc: 0.5670 | Test Acc: 0.5793  (New Best!)
Epoch 002 | LR: 0.001000 | Train Loss: 0.4406 | Val Acc: 0.7562 | Test Acc: 0.7559  (New Best!)
Epoch 003 | LR: 0.001000 | Train Loss: 0.3767 | Val Acc: 0.8286 | Test Acc: 0.8473  (New Best!)
Epoch 004 | LR: 0.001000 | Train Loss: 0.3414 | Val Acc: 0.8254
Epoch 005 | LR: 0.001000 | Train Loss: 0.3111 | Val Acc: 0.8734 | Test Acc: 0.8772  (New Best!)
Epoch 006 | LR: 0.001000 | Train Loss: 0.2935 | Val Acc: 0.8816 | Test Acc: 0.8889  (New Best!)
Epoch 007 | LR: 0.001000 | Train Loss: 0.2770 | Val Acc: 0.8694
Epoch 008 | LR: 0.001000 | Train Loss: 0.2599 | Val Acc: 0.8792
Epoch 009 | LR: 0.001000 | Train Loss: 0.2505 | Val Acc: 0.8728
Epoch 010 | LR: 0.001000 | Train Loss: 0.2377 | Val Acc: 0.9174 | Test Acc: 0.9192  (New Best!)
Epoch 011 | LR: 0.001000 | Train Loss: 0.2356 | Val Acc: 0.9232 | Test A

In [ ]:
from google.colab import drive
from torch.utils.data import Subset
from ogb.graphproppred import DglGraphPropPredDataset

drive.mount('/content/drive')
!mkdir -p /content/dataset
!unzip -q /content/drive/MyDrive/ogbg_molhiv.zip -d /content/dataset/
!ls /content/dataset/


dataset = DglGraphPropPredDataset(name='ogbg-molhiv', root='/content/dataset/')
split_idx = dataset.get_idx_split()

train_dataset = Subset(dataset, split_idx['train'])
val_dataset = Subset(dataset, split_idx['valid'])
test_dataset = Subset(dataset, split_idx['test'])

print(f"length of train set: {len(train_dataset)}, validation set: {len(val_dataset)}, test set: {len(test_dataset)}")


train_loader = GraphDataLoader(train_dataset, batch_size=128, shuffle=True, drop_last=False, num_workers=2, pin_memory=True)
val_loader = GraphDataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
test_loader = GraphDataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)


avg_log_d = compute_avg_degree(train_dataset)

Mounted at /content/drive
__MACOSX  ogbg_molhiv
length of train set: 32901, validation set: 4113, test set: 4113


In [ ]:
class BaseConfig_MOLHIV:
    name = 'MolHIV'
    epochs = 200
    batch_size = 128
    init_lr = 0.01
    lr_reduce_factor = 0.5
    min_lr = 1e-4
    weight_decay = 3e-6
    lr_schedule_patience = 20
    scheduler_mode = 'max'
    num_layers = 4
    task_dim = 1
    dropout = 0.3
    in_feat_dropout = 0.0
    use_batch_norm = True
    use_graph_norm = True
    readout = 'mean'


    edge_feat = False
    edge_dim = 0
    towers = 1
    divide_input_first = False
    divide_input_last = False
    pretrans_layers = 1
    posttrans_layers = 1



class Config_MOLHIV_PNA(BaseConfig_MOLHIV):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity', 'amplification', 'attenuation']
    hidden_dim = 70
    out_dim = 70


class Config_MOLHIV_PNA_NOSCALER(BaseConfig_MOLHIV):
    aggregators = ['mean', 'max', 'min', 'std']
    scalers = ['identity']
    hidden_dim = 95
    out_dim = 95


class Config_MOLHIV_MPNN_SUM(BaseConfig_MOLHIV):
    aggregators = ['sum']
    scalers = ['identity']
    hidden_dim = 120
    out_dim = 120


class Config_MOLHIV_MPNN_MAX(BaseConfig_MOLHIV):
    aggregators = ['max']
    scalers = ['identity']
    hidden_dim = 120
    out_dim = 120


In [ ]:
def train_epoch_molhiv(model, optimizer, dataloader, device, evaluator, config):
    model.train()
    epoch_loss = 0.0
    y_true, y_pred = [], []

    for batched_graph, labels in dataloader:
        batched_graph = batched_graph.to(device)
        labels = labels.to(device).float()

        h = batched_graph.ndata['feat']
        e = batched_graph.edata['feat'] if config.edge_feat else None

        tab_sizes = batched_graph.batch_num_nodes()
        snorm_n = [torch.ones(size, 1) / math.sqrt(size) for size in tab_sizes]
        snorm_n = torch.cat(snorm_n).to(device)

        pred = model(batched_graph, h, snorm_n, e)

        is_labeled = labels == labels
        loss = F.binary_cross_entropy_with_logits(pred[is_labeled], labels[is_labeled])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * batched_graph.batch_size
        y_true.append(labels.detach().cpu())
        y_pred.append(pred.detach().cpu())

    y_true = torch.cat(y_true, dim=0).numpy()
    y_pred = torch.cat(y_pred, dim=0).numpy()
    rocauc = evaluator.eval({'y_true': y_true, 'y_pred': y_pred})['rocauc']

    return epoch_loss / len(dataloader.dataset), rocauc


def evaluate_epoch_molhiv(model, dataloader, device, evaluator, config):
    model.eval()
    epoch_loss = 0.0
    y_true, y_pred = [], []

    with torch.no_grad():
        for batched_graph, labels in dataloader:
            batched_graph = batched_graph.to(device)
            labels = labels.to(device).float()

            h = batched_graph.ndata['feat']
            e = batched_graph.edata['feat'] if config.edge_feat else None

            tab_sizes = batched_graph.batch_num_nodes()
            snorm_n = [torch.ones(size, 1) / math.sqrt(size) for size in tab_sizes]
            snorm_n = torch.cat(snorm_n).to(device)

            pred = model(batched_graph, h, snorm_n, e)

            is_labeled = labels == labels
            loss = F.binary_cross_entropy_with_logits(pred[is_labeled], labels[is_labeled])

            epoch_loss += loss.item() * batched_graph.batch_size
            y_true.append(labels.detach().cpu())
            y_pred.append(pred.detach().cpu())

    y_true = torch.cat(y_true, dim=0).numpy()
    y_pred = torch.cat(y_pred, dim=0).numpy()
    rocauc = evaluator.eval({'y_true': y_true, 'y_pred': y_pred})['rocauc']

    return epoch_loss / len(dataloader.dataset), rocauc

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    dgl.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(41)

In [ ]:
config = Config_MOLHIV_PNA()


model = RealWorldBenchmarkModel(config, avg_log_d).to(device)
optimizer = optim.Adam(model.parameters(), lr=config.init_lr, weight_decay=config.weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=config.scheduler_mode,
    factor=config.lr_reduce_factor,
    patience=config.lr_schedule_patience,
)


evaluator = Evaluator(name='ogbg-molhiv')

best_val_metric = 0.0
test_metric_at_best_val = 0.0


for epoch in range(config.epochs):
    train_loss, train_metric = train_epoch_molhiv(model, optimizer, train_loader, device, evaluator, config)
    val_loss, val_metric = evaluate_epoch_molhiv(model, val_loader, device, evaluator, config)

    scheduler.step(val_metric)
    current_lr = optimizer.param_groups[0]['lr']

    if val_metric > best_val_metric:
        best_val_metric = val_metric
        _, test_metric = evaluate_epoch_molhiv(model, test_loader, device, evaluator, config)
        test_metric_at_best_val = test_metric
        torch.save(model.state_dict(), 'best_pna_molhiv_model.pth')
        best_mark = " (New Best!)"
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Val ROC-AUC: {val_metric:.4f} | Test ROC-AUC: {test_metric:.4f} {best_mark}")
    else:
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Val ROC-AUC: {val_metric:.4f}")

    if current_lr < config.min_lr:
        break

print(f"🏆 for Best Val : Test ROC-AUC = {test_metric_at_best_val:.4f}")

Epoch 000 | LR: 0.010000 | Train Loss: 0.2277 | Val ROC-AUC: 0.7127 | Test ROC-AUC: 0.7375  (New Best!)
Epoch 001 | LR: 0.010000 | Train Loss: 0.1532 | Val ROC-AUC: 0.7073
Epoch 002 | LR: 0.010000 | Train Loss: 0.1495 | Val ROC-AUC: 0.7543 | Test ROC-AUC: 0.7546  (New Best!)
Epoch 003 | LR: 0.010000 | Train Loss: 0.1478 | Val ROC-AUC: 0.7763 | Test ROC-AUC: 0.7358  (New Best!)
Epoch 004 | LR: 0.010000 | Train Loss: 0.1490 | Val ROC-AUC: 0.7626
Epoch 005 | LR: 0.010000 | Train Loss: 0.1426 | Val ROC-AUC: 0.7738
Epoch 006 | LR: 0.010000 | Train Loss: 0.1696 | Val ROC-AUC: 0.7716
Epoch 007 | LR: 0.010000 | Train Loss: 0.1461 | Val ROC-AUC: 0.7040
Epoch 008 | LR: 0.010000 | Train Loss: 0.1566 | Val ROC-AUC: 0.7462
Epoch 009 | LR: 0.010000 | Train Loss: 0.1456 | Val ROC-AUC: 0.7746
Epoch 010 | LR: 0.010000 | Train Loss: 0.1444 | Val ROC-AUC: 0.7550
Epoch 011 | LR: 0.010000 | Train Loss: 0.1431 | Val ROC-AUC: 0.7517
Epoch 012 | LR: 0.010000 | Train Loss: 0.1429 | Val ROC-AUC: 0.7655
Epoch 01

In [ ]:
config = Config_MOLHIV_PNA_NOSCALER()


model = RealWorldBenchmarkModel(config, avg_log_d).to(device)
optimizer = optim.Adam(model.parameters(), lr=config.init_lr, weight_decay=config.weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=config.scheduler_mode,
    factor=config.lr_reduce_factor,
    patience=config.lr_schedule_patience,
)


evaluator = Evaluator(name='ogbg-molhiv')

best_val_metric = 0.0
test_metric_at_best_val = 0.0


for epoch in range(config.epochs):
    train_loss, train_metric = train_epoch_molhiv(model, optimizer, train_loader, device, evaluator, config)
    val_loss, val_metric = evaluate_epoch_molhiv(model, val_loader, device, evaluator, config)

    scheduler.step(val_metric)
    current_lr = optimizer.param_groups[0]['lr']

    if val_metric > best_val_metric:
        best_val_metric = val_metric
        _, test_metric = evaluate_epoch_molhiv(model, test_loader, device, evaluator, config)
        test_metric_at_best_val = test_metric
        torch.save(model.state_dict(), 'best_pna_molhi_noscaler_model.pth')
        best_mark = " (New Best!)"
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Val ROC-AUC: {val_metric:.4f} | Test ROC-AUC: {test_metric:.4f} {best_mark}")
    else:
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Val ROC-AUC: {val_metric:.4f}")

    if current_lr < config.min_lr:
        break

print(f"🏆 for Best Val : Test ROC-AUC = {test_metric_at_best_val:.4f}")

Epoch 000 | LR: 0.010000 | Train Loss: 0.3131 | Val ROC-AUC: 0.6967 | Test ROC-AUC: 0.7330  (New Best!)
Epoch 001 | LR: 0.010000 | Train Loss: 0.1563 | Val ROC-AUC: 0.7324 | Test ROC-AUC: 0.7284  (New Best!)
Epoch 002 | LR: 0.010000 | Train Loss: 0.1518 | Val ROC-AUC: 0.7115
Epoch 003 | LR: 0.010000 | Train Loss: 0.1480 | Val ROC-AUC: 0.7268
Epoch 004 | LR: 0.010000 | Train Loss: 0.1470 | Val ROC-AUC: 0.7679 | Test ROC-AUC: 0.7566  (New Best!)
Epoch 005 | LR: 0.010000 | Train Loss: 0.1492 | Val ROC-AUC: 0.7483
Epoch 006 | LR: 0.010000 | Train Loss: 0.1435 | Val ROC-AUC: 0.6856
Epoch 007 | LR: 0.010000 | Train Loss: 0.1477 | Val ROC-AUC: 0.7722 | Test ROC-AUC: 0.7395  (New Best!)
Epoch 008 | LR: 0.010000 | Train Loss: 0.1441 | Val ROC-AUC: 0.7238
Epoch 009 | LR: 0.010000 | Train Loss: 0.1481 | Val ROC-AUC: 0.7813 | Test ROC-AUC: 0.7640  (New Best!)
Epoch 010 | LR: 0.010000 | Train Loss: 0.1408 | Val ROC-AUC: 0.7776
Epoch 011 | LR: 0.010000 | Train Loss: 0.1440 | Val ROC-AUC: 0.7536
Epoc

In [ ]:
config = Config_MOLHIV_MPNN_SUM()


model = RealWorldBenchmarkModel(config, avg_log_d).to(device)
optimizer = optim.Adam(model.parameters(), lr=config.init_lr, weight_decay=config.weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=config.scheduler_mode,
    factor=config.lr_reduce_factor,
    patience=config.lr_schedule_patience,
)


evaluator = Evaluator(name='ogbg-molhiv')

best_val_metric = 0.0
test_metric_at_best_val = 0.0


for epoch in range(config.epochs):
    train_loss, train_metric = train_epoch_molhiv(model, optimizer, train_loader, device, evaluator, config)
    val_loss, val_metric = evaluate_epoch_molhiv(model, val_loader, device, evaluator, config)

    scheduler.step(val_metric)
    current_lr = optimizer.param_groups[0]['lr']

    if val_metric > best_val_metric:
        best_val_metric = val_metric
        _, test_metric = evaluate_epoch_molhiv(model, test_loader, device, evaluator, config)
        test_metric_at_best_val = test_metric
        torch.save(model.state_dict(), 'best_mpnn_molhiv_sum_model.pth')
        best_mark = " (New Best!)"
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Val ROC-AUC: {val_metric:.4f} | Test ROC-AUC: {test_metric:.4f} {best_mark}")
    else:
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Val ROC-AUC: {val_metric:.4f}")

    if current_lr < config.min_lr:
        break

print(f"🏆 for Best Val : Test ROC-AUC = {test_metric_at_best_val:.4f}")

Epoch 000 | LR: 0.010000 | Train Loss: 0.2800 | Val ROC-AUC: 0.7119 | Test ROC-AUC: 0.7404  (New Best!)
Epoch 001 | LR: 0.010000 | Train Loss: 0.1598 | Val ROC-AUC: 0.7043
Epoch 002 | LR: 0.010000 | Train Loss: 0.1516 | Val ROC-AUC: 0.7259 | Test ROC-AUC: 0.7325  (New Best!)
Epoch 003 | LR: 0.010000 | Train Loss: 0.1515 | Val ROC-AUC: 0.7390 | Test ROC-AUC: 0.7395  (New Best!)
Epoch 004 | LR: 0.010000 | Train Loss: 0.1657 | Val ROC-AUC: 0.7187
Epoch 005 | LR: 0.010000 | Train Loss: 0.1495 | Val ROC-AUC: 0.7194
Epoch 006 | LR: 0.010000 | Train Loss: 0.1512 | Val ROC-AUC: 0.7117
Epoch 007 | LR: 0.010000 | Train Loss: 0.1481 | Val ROC-AUC: 0.7588 | Test ROC-AUC: 0.7426  (New Best!)
Epoch 008 | LR: 0.010000 | Train Loss: 0.1566 | Val ROC-AUC: 0.7284
Epoch 009 | LR: 0.010000 | Train Loss: 0.1505 | Val ROC-AUC: 0.7631 | Test ROC-AUC: 0.7073  (New Best!)
Epoch 010 | LR: 0.010000 | Train Loss: 0.1505 | Val ROC-AUC: 0.7482
Epoch 011 | LR: 0.010000 | Train Loss: 0.1478 | Val ROC-AUC: 0.7713 | Te

In [ ]:
config = Config_MOLHIV_MPNN_MAX()


model = RealWorldBenchmarkModel(config, avg_log_d).to(device)
optimizer = optim.Adam(model.parameters(), lr=config.init_lr, weight_decay=config.weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=config.scheduler_mode,
    factor=config.lr_reduce_factor,
    patience=config.lr_schedule_patience,
)


evaluator = Evaluator(name='ogbg-molhiv')

best_val_metric = 0.0
test_metric_at_best_val = 0.0


for epoch in range(config.epochs):
    train_loss, train_metric = train_epoch_molhiv(model, optimizer, train_loader, device, evaluator, config)
    val_loss, val_metric = evaluate_epoch_molhiv(model, val_loader, device, evaluator, config)

    scheduler.step(val_metric)
    current_lr = optimizer.param_groups[0]['lr']

    if val_metric > best_val_metric:
        best_val_metric = val_metric
        _, test_metric = evaluate_epoch_molhiv(model, test_loader, device, evaluator, config)
        test_metric_at_best_val = test_metric
        torch.save(model.state_dict(), 'best_mpnn_molhiv_max_model.pth')
        best_mark = " (New Best!)"
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Val ROC-AUC: {val_metric:.4f} | Test ROC-AUC: {test_metric:.4f} {best_mark}")
    else:
        print(f"Epoch {epoch:03d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Val ROC-AUC: {val_metric:.4f}")

    if current_lr < config.min_lr:
        break

print(f"🏆 for Best Val : Test ROC-AUC = {test_metric_at_best_val:.4f}")

Epoch 000 | LR: 0.010000 | Train Loss: 0.6035 | Val ROC-AUC: 0.6438 | Test ROC-AUC: 0.6387  (New Best!)
Epoch 001 | LR: 0.010000 | Train Loss: 0.1613 | Val ROC-AUC: 0.6850 | Test ROC-AUC: 0.7206  (New Best!)
Epoch 002 | LR: 0.010000 | Train Loss: 0.1511 | Val ROC-AUC: 0.5946
Epoch 003 | LR: 0.010000 | Train Loss: 0.1505 | Val ROC-AUC: 0.7478 | Test ROC-AUC: 0.7149  (New Best!)
Epoch 004 | LR: 0.010000 | Train Loss: 0.1469 | Val ROC-AUC: 0.7777 | Test ROC-AUC: 0.7231  (New Best!)
Epoch 005 | LR: 0.010000 | Train Loss: 0.1453 | Val ROC-AUC: 0.7453
Epoch 006 | LR: 0.010000 | Train Loss: 0.1459 | Val ROC-AUC: 0.7645
Epoch 007 | LR: 0.010000 | Train Loss: 0.1473 | Val ROC-AUC: 0.7790 | Test ROC-AUC: 0.7568  (New Best!)
Epoch 008 | LR: 0.010000 | Train Loss: 0.1426 | Val ROC-AUC: 0.7932 | Test ROC-AUC: 0.7456  (New Best!)
Epoch 009 | LR: 0.010000 | Train Loss: 0.1412 | Val ROC-AUC: 0.7961 | Test ROC-AUC: 0.7451  (New Best!)
Epoch 010 | LR: 0.010000 | Train Loss: 0.1409 | Val ROC-AUC: 0.7360
